# TMD Band Generator

Generate a `k_T`-space TMD band `.pkl` directly from replica-parameter CSVs.

For each parameter row:
- if `pdf_replica_id` is present, use the matching `TMDPDF_plot/<id>.csv` grid
- if it is missing, assign a random available grid member
- evaluate the `k_T`-space curves with the current NP parameters
- collect central, mean, median, and central-68% bands into one pickle payload


In [1]:
from pathlib import Path

fit_name = "Final_old"
#replica_results_path = Path("../Fits/replica_data/replica_0325.csv")
#output_band_path = Path("TMD bands") / "replica_tmd_band.pkl"
replica_results_path = Path("../Bayesian Analysis/bayesian parameters/bayesian_parameters_3000.csv")
output_band_path = Path("TMD bands") / "bayesian_tmd_band.pkl"

use_success_only = True
max_replicas = None
band_alpha = 15.865
central_curve_mode = "card"  # "card" or "replica_mean"
random_pdf_seed = 12345

map_plot_flavor = "u"
map_x_values = [1e-3, 1e-2, 1e-1]
map_Q_values = [5.0, 100.0]

kt_plot_min = 0.0
kt_plot_max = 3.0
n_kt_plot = 50

FLAVORS = ["u", "ub", "d", "db", "s", "sb", "c", "cb", "b", "bb"]
FLAVOR_INDEX = {name: i for i, name in enumerate(FLAVORS)}
flavor_idx = FLAVOR_INDEX[map_plot_flavor]

replica_results_path = replica_results_path.resolve()
output_band_path = output_band_path.resolve()
output_band_path.parent.mkdir(parents=True, exist_ok=True)


In [2]:
import csv
import pickle
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from julia import Main
from tqdm.auto import tqdm

repo_root = Path("..").resolve()
TMD_root = (repo_root / "TMDs").resolve()
card_path = repo_root / "Cards" / f"{fit_name}.jl"
card_text = card_path.read_text(encoding="utf-8")

struct_match = re.search(r"struct\s+Params_Struct(.*?)end", card_text, re.S)
if struct_match is None:
    raise ValueError(f"Could not find Params_Struct in {card_path}")

param_names = re.findall(r"([A-Za-z_][A-Za-z0-9_]*)\s*::\s*Float32", struct_match.group(1))
param_columns = [f"param_{i}" for i in range(len(param_names))]

init_matches = re.findall(r"(?ms)^\s*initial_params\s*=\s*\[([^\]]*)\]", card_text)
if not init_matches:
    raise ValueError(f"Could not find initial_params in {card_path}")
initial_params = np.asarray(
    [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", init_matches[-1])],
    dtype=float,
)


def load_replica_results(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(f"{path} does not contain any replica rows")

    working = df.copy()

    if all(col in working.columns for col in param_columns):
        for col in param_columns:
            working[col] = pd.to_numeric(working[col], errors="coerce")
        for col, name in zip(param_columns, param_names):
            working[name] = working[col]
    elif all(name in working.columns for name in param_names):
        for name in param_names:
            working[name] = pd.to_numeric(working[name], errors="coerce")
        for col, name in zip(param_columns, param_names):
            working[col] = working[name]
    else:
        raise ValueError(
            f"{path} must contain either {param_columns} or the named parameter columns {param_names}"
        )

    if "replica_id" not in working.columns:
        if "source_index" in working.columns:
            working["replica_id"] = pd.to_numeric(working["source_index"], errors="coerce")
        else:
            working["replica_id"] = np.arange(len(working), dtype=int)
    if "success" not in working.columns:
        working["success"] = 1
    if "nfev" not in working.columns:
        working["nfev"] = pd.Series(pd.array([pd.NA] * len(working), dtype="Int64"), index=working.index)
    if "best_chi2dN" not in working.columns:
        working["best_chi2dN"] = np.nan
    if "source_index" in working.columns:
        working["source_index"] = pd.to_numeric(working["source_index"], errors="coerce").astype("Int64")
    if "log_prob" in working.columns:
        working["log_prob"] = pd.to_numeric(working["log_prob"], errors="coerce")

    if "pdf_replica_id" not in working.columns:
        working["pdf_replica_id"] = pd.Series(pd.array([pd.NA] * len(working), dtype="Int64"), index=working.index)
    else:
        working["pdf_replica_id"] = pd.to_numeric(working["pdf_replica_id"], errors="coerce").astype("Int64")

    for col in ["replica_id", "success", "nfev", "best_chi2dN"]:
        working[col] = pd.to_numeric(working[col], errors="coerce")

    working = working.dropna(subset=["replica_id", *param_columns]).copy()
    working["replica_id"] = working["replica_id"].astype(int)
    working["success"] = pd.to_numeric(working["success"], errors="coerce").astype("Int64")
    working["nfev"] = pd.to_numeric(working["nfev"], errors="coerce").astype("Int64")

    return working.sort_values("replica_id").reset_index(drop=True)


replica_results_df = load_replica_results(replica_results_path)
if use_success_only:
    replica_results_df = replica_results_df[replica_results_df["success"] == 1].reset_index(drop=True)
if max_replicas is not None:
    replica_results_df = replica_results_df.iloc[:max_replicas].copy()
if replica_results_df.empty:
    raise ValueError("No replica rows available after filtering")


def include(name: str) -> None:
    path = (repo_root / name).resolve()
    Main.eval(f'include(raw"{path}")')


include(f"Cards/{fit_name}.jl")


def Push_Params(params: np.ndarray) -> None:
    lines = []
    for name, val in zip(param_names, params):
        lines.append(f"global NP_{name} = Float32({float(val)})")
    Main.eval("\n".join(lines))


Push_Params(initial_params)

active_table_name = str(Main.table_name)
np_cl_name = str(Main.NP_name)
np_julia_name = Path(np_cl_name).with_suffix(".jl").name
np_julia_path = TMD_root / "NP Parameterizations Julia" / np_julia_name
if not np_julia_path.exists():
    raise FileNotFoundError(f"No Julia NP file found for {np_cl_name}. Expected {np_julia_path.name}.")

Main.if_grid = True
try:
    Main.FastGK
except Exception:
    Main.eval("module FastGK end")
try:
    Main.eval("b0")
except Exception:
    Main.eval("const b0 = 1.1229189")

include("TMDs/Grids/initialization.jl")
include(f"TMDs/NP Parameterizations Julia/{np_julia_name}")
Push_Params(initial_params)
include("TMDs/TMDPDFs/TMDPDFN.jl")
include("Replica Analysis/TMD_band_generator.jl")

default_tmdpdf_grid_path = TMD_root / "Grids" / active_table_name / "TMDPDF.csv"
tmdpdf_plot_dir = TMD_root / "Grids" / active_table_name / "TMDPDF_plot"
if not default_tmdpdf_grid_path.exists():
    raise FileNotFoundError(f"Missing default TMDPDF grid: {default_tmdpdf_grid_path}")
if not tmdpdf_plot_dir.exists():
    raise FileNotFoundError(f"Missing TMDPDF_plot directory: {tmdpdf_plot_dir}")

available_pdf_grid_paths = {
    int(path.stem): path.resolve()
    for path in sorted(tmdpdf_plot_dir.glob("*.csv"), key=lambda p: int(p.stem))
    if path.stem.isdigit()
}
if not available_pdf_grid_paths:
    raise FileNotFoundError(f"No numeric TMDPDF_plot/*.csv grids found under {tmdpdf_plot_dir}")

print(f"fit_name = {fit_name}")
print(f"table_name = {active_table_name}")
print(f"replicas used = {len(replica_results_df)}")
print(f"available pdf grids = {len(available_pdf_grid_paths)}")
display(pd.DataFrame({"param": param_names, "central_value": initial_params}))
preview_cols = ["replica_id", "pdf_replica_id", "success", "best_chi2dN"]
for col in ["source_index", "log_prob"]:
    if col in replica_results_df.columns:
        preview_cols.append(col)
display(replica_results_df[preview_cols].head())


c:\Users\congyue zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


fit_name = Final_old
table_name = MSHT20N3LO-MC-4-2
replicas used = 3000
available pdf grids = 100


,param,central_value
0,lambda1,0.023656
1,lambda2,1.054291
2,lambda3,-2.354365
3,logx0,-5.207703
4,sigx,1.103274
5,amp,-0.431106
6,BNP,1.494665
7,c0,0.070013
8,c1,0.027637


,replica_id,pdf_replica_id,success,best_chi2dN,source_index,log_prob
0,9,<NA>,1,NaN,9,-186.607928
1,31,<NA>,1,NaN,31,-186.622503
2,41,<NA>,1,NaN,41,-184.315792
3,51,<NA>,1,NaN,51,-185.509553
4,77,<NA>,1,NaN,77,-185.064431


In [3]:
TMDPDF_TARGET_NAMES = ["f_u", "f_ub", "f_d", "f_db", "f_s", "f_sb", "f_c", "f_cb", "f_b", "f_bb"]
JULIA_TMDPDF_TARGET_NAMES = "[" + ", ".join(f'\"{name}\"' for name in TMDPDF_TARGET_NAMES) + "]"
current_pdf_grid_label = None
current_pdf_grid_path = None


def set_tmdpdf_grid_from_path(grid_path: Path, grid_label) -> None:
    global current_pdf_grid_label, current_pdf_grid_path

    grid_path = Path(grid_path).resolve()
    if current_pdf_grid_path == grid_path:
        return

    Main.eval(
        f'''
        begin
            local df = DataFrame(CSV.File(raw"{grid_path}"))
            global df_TMDPDF = df
            global TMDPDF_bmin = Float64(minimum(df[!, "b"]))
            global TMDPDF_bmax = Float64(maximum(df[!, "b"]))
            initialize_interpolator(
                df = df,
                interpolator_name = "xTMDPDF_raw_grid",
                variable_names = ["x", "b"],
                target_names = {JULIA_TMDPDF_TARGET_NAMES},
            )
            let itp = interpolators[:xTMDPDF_raw_grid]
                global xTMDPDF_raw_grid
                @inline xTMDPDF_raw_grid(x::Real, b::Real) = itp(x, b)
            end
        end
        '''
    )
    current_pdf_grid_label = grid_label
    current_pdf_grid_path = grid_path


def restore_default_tmdpdf_grid() -> None:
    set_tmdpdf_grid_from_path(default_tmdpdf_grid_path, "default")


def set_tmdpdf_grid_from_member(pdf_replica_id: int) -> None:
    if int(pdf_replica_id) not in available_pdf_grid_paths:
        raise KeyError(
            f"No TMDPDF_plot grid found for pdf_replica_id={int(pdf_replica_id)} under {tmdpdf_plot_dir}"
        )
    set_tmdpdf_grid_from_path(available_pdf_grid_paths[int(pdf_replica_id)], int(pdf_replica_id))


def assign_effective_pdf_replica_ids(df: pd.DataFrame) -> pd.DataFrame:
    rng = np.random.default_rng(random_pdf_seed)
    available_ids = np.array(sorted(available_pdf_grid_paths), dtype=int)
    working = df.copy()

    effective_ids = []
    assignment_sources = []
    input_ids = []

    for value in working["pdf_replica_id"]:
        input_ids.append(pd.NA if pd.isna(value) else int(value))
        if pd.isna(value):
            effective_id = int(rng.choice(available_ids))
            assignment_source = "random"
        else:
            effective_id = int(value)
            assignment_source = "csv"
            if effective_id not in available_pdf_grid_paths:
                raise KeyError(
                    f"Row requested pdf_replica_id={effective_id}, but no matching grid exists in {tmdpdf_plot_dir}"
                )
        effective_ids.append(effective_id)
        assignment_sources.append(assignment_source)

    working["input_pdf_replica_id"] = pd.Series(pd.array(input_ids, dtype="Int64"), index=working.index)
    working["effective_pdf_replica_id"] = pd.Series(effective_ids, dtype="Int64", index=working.index)
    working["pdf_assignment_source"] = assignment_sources
    return working


def as_array(julia_tuple) -> np.ndarray:
    return np.asarray(julia_tuple, dtype=float)


def evaluate_curve_bundle(params: np.ndarray, kt_grid: np.ndarray, x_values, q_values):
    Push_Params(params)
    kt_curves = {}
    for q_value in q_values:
        for x_value in x_values:
            key = (float(q_value), float(x_value))
            values = as_array(
                Main.TMDPDF_kt_vec(
                    kt_vec=np.asarray(kt_grid, dtype=np.float64),
                    x=float(x_value),
                    Q=float(q_value),
                )
            )
            kt_curves[key] = np.asarray(x_value * values[flavor_idx, :], dtype=float)
    return kt_curves


def build_space_payload(grid: np.ndarray, axis_name: str, stats_map: dict, central_curves: dict, q_values, x_values):
    payload = {}
    for q_value in q_values:
        q_dict = {}
        for x_value in x_values:
            key = (float(q_value), float(x_value))
            stats = stats_map[key]
            q_dict[float(x_value)] = pd.DataFrame(
                {
                    axis_name: grid,
                    "central": central_curves[key],
                    "pred_mean": stats["pred_mean"],
                    "pred_median": stats["pred_median"],
                    "pred_low": stats["pred_low"],
                    "pred_up": stats["pred_up"],
                    "pred_lo": stats["pred_low"],
                    "pred_hi": stats["pred_up"],
                    "pred_mid": stats["pred_median"],
                }
            )
        payload[float(q_value)] = q_dict
    return payload


replica_results_df = assign_effective_pdf_replica_ids(replica_results_df)
print("PDF grid assignment sources:")
display(replica_results_df["pdf_assignment_source"].value_counts().rename("count").to_frame())
display(
    replica_results_df[
        [
            "replica_id",
            "input_pdf_replica_id",
            "effective_pdf_replica_id",
            "pdf_assignment_source",
            "success",
            "best_chi2dN",
        ]
    ].head()
)


PDF grid assignment sources:


,count
pdf_assignment_source,
random,3000


,replica_id,input_pdf_replica_id,effective_pdf_replica_id,pdf_assignment_source,success,best_chi2dN
0,9,<NA>,70,random,1,NaN
1,31,<NA>,23,random,1,NaN
2,41,<NA>,79,random,1,NaN
3,51,<NA>,32,random,1,NaN
4,77,<NA>,21,random,1,NaN


In [4]:
kt_plot = np.linspace(kt_plot_min, kt_plot_max, n_kt_plot)
curve_keys = [(float(q_value), float(x_value)) for q_value in map_Q_values for x_value in map_x_values]

restore_default_tmdpdf_grid()
card_kt_curves = evaluate_curve_bundle(initial_params, kt_plot, map_x_values, map_Q_values)

replica_kt_samples = {key: [] for key in curve_keys}
assignment_df = replica_results_df.copy().reset_index(drop=True)
start_time = time.perf_counter()

for row in tqdm(
    assignment_df.itertuples(index=False),
    total=len(assignment_df),
    desc="Evaluating TMD rows",
):
    params = np.asarray([getattr(row, col) for col in param_columns], dtype=float)
    set_tmdpdf_grid_from_member(int(row.effective_pdf_replica_id))
    kt_curves = evaluate_curve_bundle(params, kt_plot, map_x_values, map_Q_values)
    for key in curve_keys:
        replica_kt_samples[key].append(np.asarray(kt_curves[key], dtype=float))

restore_default_tmdpdf_grid()
replica_kt_arrays = {key: np.stack(replica_kt_samples[key], axis=0) for key in curve_keys}

def compute_band_from_array(arr: np.ndarray):
    pred_low, pred_median, pred_up = np.quantile(
        arr,
        [band_alpha / 100.0, 0.5, 1.0 - band_alpha / 100.0],
        axis=0,
    )
    return {
        "pred_mean": np.mean(arr, axis=0),
        "pred_median": pred_median,
        "pred_low": pred_low,
        "pred_up": pred_up,
    }

kt_stats = {key: compute_band_from_array(replica_kt_arrays[key]) for key in curve_keys}
replica_mean_kt_curves = {key: kt_stats[key]["pred_mean"] for key in curve_keys}

if central_curve_mode == "card":
    central_kt_curves = card_kt_curves
elif central_curve_mode == "replica_mean":
    central_kt_curves = replica_mean_kt_curves
else:
    raise ValueError(f"Unknown central_curve_mode={central_curve_mode!r}; use 'card' or 'replica_mean'")

elapsed_seconds = time.perf_counter() - start_time

band_payload = {
    "metadata": {
        "fit_name": fit_name,
        "card_path": str(card_path),
        "replica_results_path": str(replica_results_path),
        "output_band_path": str(output_band_path),
        "table_name": active_table_name,
        "flavor": map_plot_flavor,
        "band_alpha": float(band_alpha),
        "central_curve_mode": central_curve_mode,
        "random_pdf_seed": int(random_pdf_seed),
        "evaluation_mode": "direct",
        "map_x_values": [float(x) for x in map_x_values],
        "map_Q_values": [float(q) for q in map_Q_values],
        "kt_grid_size": int(len(kt_plot)),
        "n_replicas": int(len(replica_results_df)),
        "evaluation_time_seconds": float(elapsed_seconds),
        "available_pdf_grid_ids": sorted(int(x) for x in available_pdf_grid_paths),
    },
    "replica_info": assignment_df,
    "kt_space": build_space_payload(kt_plot, "kt", kt_stats, central_kt_curves, map_Q_values, map_x_values),
}

with output_band_path.open("wb") as handle:
    pickle.dump(band_payload, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Wrote TMD band payload to {output_band_path}") 
print(f"Elapsed evaluation time: {elapsed_seconds:.1f} s")
display(assignment_df.head())
display(band_payload["kt_space"][float(map_Q_values[0])][float(map_x_values[0])].head())


Evaluating TMD rows: 100%|██████████| 3000/3000 [11:46<00:00,  4.25it/s]


Wrote TMD band payload to C:\Users\congyue zhang\Desktop\OpenCL fitter\TMD-Fits-Minimal\Replica Analysis\TMD bands\bayesian_tmd_band.pkl
Elapsed evaluation time: 706.9 s


,source_index,log_prob,lambda1,lambda2,lambda3,logx0,sigx,amp,BNP,c0,...,param_7,param_8,replica_id,success,nfev,best_chi2dN,pdf_replica_id,input_pdf_replica_id,effective_pdf_replica_id,pdf_assignment_source
0,9,-186.607928,0.004001,1.066987,-2.639572,-5.354135,1.032392,-0.464033,1.915794,0.059724,...,0.059724,0.022447,9,1,<NA>,NaN,<NA>,<NA>,70,random
1,31,-186.622503,0.004191,1.143977,-2.681464,-5.042142,1.527820,-0.423431,1.922110,0.062913,...,0.062913,0.023972,31,1,<NA>,NaN,<NA>,<NA>,23,random
2,41,-184.315792,0.002548,1.122492,-2.657517,-5.301107,1.660783,-0.429345,1.741697,0.064500,...,0.064500,0.026654,41,1,<NA>,NaN,<NA>,<NA>,79,random
3,51,-185.509553,-0.008229,1.084817,-2.818570,-5.745763,1.447606,-0.570028,1.801397,0.065770,...,0.065770,0.027093,51,1,<NA>,NaN,<NA>,<NA>,32,random
4,77,-185.064431,0.001206,1.091165,-2.826395,-5.566091,1.262800,-0.494137,1.995701,0.063111,...,0.063111,0.024153,77,1,<NA>,NaN,<NA>,<NA>,21,random


,kt,central,pred_mean,pred_median,pred_low,pred_up,pred_lo,pred_hi,pred_mid
0,0.000000,0.113602,0.108620,0.107180,0.094329,0.122795,0.094329,0.122795,0.107180
1,0.061224,0.113112,0.108192,0.106794,0.094017,0.122276,0.094017,0.122276,0.106794
2,0.122449,0.111663,0.106924,0.105568,0.093035,0.120699,0.093035,0.120699,0.105568
3,0.183673,0.109320,0.104867,0.103642,0.091510,0.118138,0.091510,0.118138,0.103642
4,0.244898,0.106177,0.102101,0.101035,0.089305,0.114656,0.089305,0.114656,0.101035
